# Learned Stage-Wise Curvature Tuning on Beans

This notebook evaluates Learned Stage-Wise Curvature Tuning (SW-CT) on the Beans image-classification dataset using ImageNet-pretrained ResNet models.

The experiments compare four conditions:

1. **Baseline** — the original pretrained ResNet with ReLU activations and a trained downstream classifier.
2. **Single-Parameter Curvature Tuning (S-CT)** — the original Curvature Tuning method, which searches for one global beta value using validation performance.
3. **Learned SW-CT with fixed initialization** — four trainable stage-wise beta parameters, all initialized at beta = 0.78.
4. **Learned SW-CT with random initialization** — four trainable stage-wise beta parameters independently initialized between 0.70 and 0.99.

The purpose of testing both fixed and random initialization is to determine whether the performance of Learned SW-CT depends on the previously selected beta = 0.78 starting point.

Each method is evaluated using seeds 42, 43, and 44.

Experiments are first conducted for 20 epochs and then repeated for 30 epochs to determine whether additional optimization improves accuracy or changes the learned stage-wise curvature pattern.

Wall-clock execution time is also recorded for every method to compare computational cost.

The experiments are repeated using:

- ResNet-18
- ResNet-50
- ResNet-152

In [1]:
!nvidia-smi

Wed Sep 30 22:54:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%cd /content

!git clone -b stage-wise-ct \
    https://github.com/Syed1611/curvature-tuning-research.git

%cd /content/curvature-tuning-research/src/curvature-tuning

!git branch --show-current
!git log -1 --oneline

/content
Cloning into 'curvature-tuning-research'...
remote: Enumerating objects: 563, done.
remote: Counting objects: 100% (235/235), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 563 (delta 147), reused 179 (delta 102), pack-reused 328 (from 1)
Receiving objects: 100% (563/563), 6.52 MiB | 7.07 MiB/s, done.
Resolving deltas: 100% (278/278), done.
/content/curvature-tuning-research/src/curvature-tuning
stage-wise-ct
c0c969f (HEAD -> stage-wise-ct, origin/stage-wise-ct, origin/HEAD) updated OCTMNIST notebook


## Environment Setup

The repository dependencies are installed before running the experiments.

After installation, the Colab runtime should be restarted so that the required package versions are loaded correctly.

In [3]:
%pip install -q -r requirements.txt
%pip install --force-reinstall --no-cache-dir "numpy==1.26.3"

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 8.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 37.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.2/133.2 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 111.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 117.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 93.2 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.

In [2]:
%cd /content/curvature-tuning-research/src/curvature-tuning

import os
import torch

os.environ["WANDB_MODE"] = "disabled"

print("Working directory:", os.getcwd())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/content/curvature-tuning-research/src/curvature-tuning
Working directory: /content/curvature-tuning-research/src/curvature-tuning
PyTorch: 2.6.0+cu124
CUDA available: True
GPU: Tesla T4


## Beans Dataset

Before running the experiments, the Beans data loaders are checked to confirm that the downstream dataset can be loaded correctly.

The same dataset split and random seed protocol are used for Baseline, S-CT, and Learned SW-CT so that the methods are compared under the same conditions.

In [3]:
from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_beans",
    train_batch_size=32,
    test_batch_size=800,
    seed=42,
)

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels[:10])

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


data/train-00000-of-00001.parquet:   0%|          | 0.00/144M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/18.5M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Train samples: 1034
Validation samples: 133
Test samples: 128
Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Labels: tensor([1, 1, 0, 1, 0, 2, 1, 0, 1, 1])


# ResNet-18 Experiments

ResNet-18 is evaluated first because it was used during the initial development of Learned SW-CT.

The experiment compares Baseline, S-CT, fixed-initialization SW-CT, and random-initialization SW-CT.

All methods use the same ImageNet-pretrained backbone, Beans dataset split, experimental seeds, and downstream training protocol.

## ResNet-18 — 20-Epoch Evaluation

The first evaluation uses 20 training epochs.

This provides the primary reference experiment and maintains compatibility with the earlier Learned SW-CT work.

### Methods

**Baseline**

The pretrained ResNet-18 retains its original ReLU activations. The backbone is frozen and only the downstream classifier is trained.

**S-CT**

The original Single-Parameter Curvature Tuning method searches over a single beta shared throughout the network. Candidate beta values from 0.70 to 1.00 are evaluated using validation performance, and the best candidate is evaluated on the test set.

The recorded S-CT execution time includes the complete beta search rather than only the final selected beta.

**Learned SW-CT — Fixed Initialization**

Four trainable beta parameters are assigned to the four ResNet stages. All four begin at beta = 0.78.

This preserves the initialization used in the original Learned SW-CT experiments.

**Learned SW-CT — Random Initialization**

Four trainable stage-wise beta parameters are initialized independently between 0.70 and 0.99.

The random initialization is generated deterministically from the experiment seed. This allows the experiment to test whether Learned SW-CT depends on the fixed 0.78 starting value while remaining reproducible.

In [4]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 800 \
    --epochs 20

2026-09-30 22:59:08.665 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_beans_resnet18_seed42_epochs20.log
2026-09-30 22:59:08.672 | INFO     | __main__:main:78 - Running on cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 159MB/s] 
2026-09-30 22:59:10.227 | INFO     | __main__:main:105 - Testing baseline...
2026-09-30 22:59:10.244 | INFO     | __main__:main:116 - Number of trainable parameters: 1539
2026-09-30 22:59:10.244 | INFO     | __main__:main:117 - Starting transfer learning...
2026-09-30 22:59:19.862 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.203847, Accuracy: 31.25%
2026-09-30 22:59:20.174 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.816237, Val Accuracy: 69.92%
2026-09-30 22:59:20.175 | INFO     | train:linear_probe:190 - New best validation accuracy: 69.92 at epoch 1
2026-09-30 22:59:20.233 | INFO

In [5]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 800 \
    --epochs 20

2026-09-30 23:06:59.688 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_beans_resnet18_seed43_epochs20.log
2026-09-30 23:06:59.692 | INFO     | __main__:main:78 - Running on cuda
2026-09-30 23:07:00.805 | INFO     | __main__:main:105 - Testing baseline...
2026-09-30 23:07:00.820 | INFO     | __main__:main:116 - Number of trainable parameters: 1539
2026-09-30 23:07:00.821 | INFO     | __main__:main:117 - Starting transfer learning...
2026-09-30 23:07:09.823 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.063459, Accuracy: 50.00%
2026-09-30 23:07:10.045 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.718151, Val Accuracy: 75.94%
2026-09-30 23:07:10.046 | INFO     | train:linear_probe:190 - New best validation accuracy: 75.94 at epoch 1
2026-09-30 23:07:10.115 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 0.651616, Accuracy: 90.62%
2026-09-30 23:07:10.335 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 0.499973, Val Ac

In [6]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 800 \
    --epochs 20

2026-09-30 23:14:57.143 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_beans_resnet18_seed44_epochs20.log
2026-09-30 23:14:57.147 | INFO     | __main__:main:78 - Running on cuda
2026-09-30 23:14:58.253 | INFO     | __main__:main:105 - Testing baseline...
2026-09-30 23:14:58.268 | INFO     | __main__:main:116 - Number of trainable parameters: 1539
2026-09-30 23:14:58.269 | INFO     | __main__:main:117 - Starting transfer learning...
2026-09-30 23:15:07.289 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.341334, Accuracy: 46.88%
2026-09-30 23:15:07.520 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.846501, Val Accuracy: 61.65%
2026-09-30 23:15:07.522 | INFO     | train:linear_probe:190 - New best validation accuracy: 61.65 at epoch 1
2026-09-30 23:15:07.584 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 0.886848, Accuracy: 56.25%
2026-09-30 23:15:07.809 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 0.560702, Val Ac

### Learned SW-CT with Fixed Beta Initialization

The original Learned SW-CT configuration is first reproduced using the fixed initialization:

\[[\beta_1,\beta_2,\beta_3,\beta_4]=[0.78,0.78,0.78,0.78].\]

The initial value is only a starting point. During training, the four beta parameters are optimized independently together with the downstream classifier.

These runs provide the reference against which random initialization will be compared.

In [7]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-09-30 23:22:58.592 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed42_epochs20_initfixed0.78_betalr0.01.log
2026-09-30 23:22:58.664 | INFO     | __main__:main:302 - Running on cuda
2026-09-30 23:22:59.770 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-09-30 23:22:59.770 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-30 23:22:59.795 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-09-30 23:22:59.796 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-09-30 23:22:59.796 | INFO     | __main__:main:439 - Total trainable parameters: 1543
2026-09-30 23:23:01.862 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.032128, Accuracy: 46.88%
2026-09-30 23:23:09.891 | INFO     | train:test_epoch:90 - Epo

In [8]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-09-30 23:26:20.184 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed43_epochs20_initfixed0.78_betalr0.01.log
2026-09-30 23:26:20.253 | INFO     | __main__:main:302 - Running on cuda
2026-09-30 23:26:21.372 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-09-30 23:26:21.373 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-30 23:26:21.401 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-09-30 23:26:21.402 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-09-30 23:26:21.402 | INFO     | __main__:main:439 - Total trainable parameters: 1543
2026-09-30 23:26:23.322 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.324863, Accuracy: 28.12%
2026-09-30 23:26:31.629 | INFO     | train:test_epoch:90 - Epo

In [9]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-09-30 23:29:40.868 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed44_epochs20_initfixed0.78_betalr0.01.log
2026-09-30 23:29:40.935 | INFO     | __main__:main:302 - Running on cuda
2026-09-30 23:29:41.994 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-09-30 23:29:41.995 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-30 23:29:42.021 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-09-30 23:29:42.021 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-09-30 23:29:42.022 | INFO     | __main__:main:439 - Total trainable parameters: 1543
2026-09-30 23:29:43.383 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.201310, Accuracy: 31.25%
2026-09-30 23:29:52.440 | INFO     | train:test_epoch:90 - Epo

### Learned SW-CT with Random Beta Initialization

The fixed beta = 0.78 initialization may raise the question of whether Learned SW-CT performs well only because its starting point was informed by the earlier S-CT experiments.

To test this dependency, each of the four stage-wise beta parameters is now initialized independently from a uniform distribution between 0.70 and 0.99.

The initialization is reproducible: the random generator is seeded using the experiment seed. Therefore, each seed always receives the same initial beta vector.

If random initialization produces similar performance to the fixed 0.78 experiment, this provides evidence that the Learned SW-CT result is not dependent on a single hand-selected initialization.

In [10]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-09-30 23:33:02.340 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed42_epochs20_initrandom_betalr0.01.log
2026-09-30 23:33:02.409 | INFO     | __main__:main:302 - Running on cuda
2026-09-30 23:33:03.522 | INFO     | __main__:main:389 - Initialization strategy: random
2026-09-30 23:33:03.522 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-30 23:33:03.547 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-09-30 23:33:03.548 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-09-30 23:33:03.548 | INFO     | __main__:main:439 - Total trainable parameters: 1543
2026-09-30 23:33:04.913 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.046550, Accuracy: 46.88%
2026-09-30 23:33:14.002 | INFO     | train:test_ep

In [11]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-09-30 23:36:23.186 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed43_epochs20_initrandom_betalr0.01.log
2026-09-30 23:36:23.252 | INFO     | __main__:main:302 - Running on cuda
2026-09-30 23:36:24.322 | INFO     | __main__:main:389 - Initialization strategy: random
2026-09-30 23:36:24.322 | INFO     | __main__:main:394 - Requested initial stage betas: [0.889167, 0.712695, 0.705809, 0.943372]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-30 23:36:24.347 | INFO     | __main__:main:418 - Initial stage betas: [0.8891667127609253, 0.7126947641372681, 0.7058085799217224, 0.9433716535568237]
2026-09-30 23:36:24.348 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-09-30 23:36:24.348 | INFO     | __main__:main:439 - Total trainable parameters: 1543
2026-09-30 23:36:25.617 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.278397, Accuracy: 28.12%
2026-09-30 23:36:34.531 | INFO     | train:test_

In [12]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-09-30 23:39:40.615 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed44_epochs20_initrandom_betalr0.01.log
2026-09-30 23:39:40.682 | INFO     | __main__:main:302 - Running on cuda
2026-09-30 23:39:41.699 | INFO     | __main__:main:389 - Initialization strategy: random
2026-09-30 23:39:41.700 | INFO     | __main__:main:394 - Requested initial stage betas: [0.735544, 0.774853, 0.817674, 0.981063]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-30 23:39:41.724 | INFO     | __main__:main:418 - Initial stage betas: [0.7355440258979797, 0.7748528122901917, 0.8176735043525696, 0.9810633659362793]
2026-09-30 23:39:41.725 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-09-30 23:39:41.725 | INFO     | __main__:main:439 - Total trainable parameters: 1543
2026-09-30 23:39:43.100 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.202614, Accuracy: 34.38%
2026-09-30 23:39:51.698 | INFO     | train:test_

# ResNet-18 — 30-Epoch Evaluation

The same experiments are now repeated for 30 epochs.

The purpose of increasing the training budget is to determine whether the 20-epoch results are limited by insufficient optimization.

If a method improves substantially with 30 epochs, this suggests that additional optimization is useful. If performance remains similar, it indicates that the method has largely stabilized within the original 20-epoch training budget.

All other experimental settings remain unchanged.

For random-initialization SW-CT, the same seed produces exactly the same initial four-beta vector in both the 20-epoch and 30-epoch experiments. Therefore, initialization is held constant while the training duration changes.

Execution time is again recorded to measure the computational cost of extending training from 20 to 30 epochs.

In [13]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 800 \
    --epochs 30

2026-09-30 23:42:58.139 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_beans_resnet18_seed42_epochs30.log
2026-09-30 23:42:58.143 | INFO     | __main__:main:78 - Running on cuda
2026-09-30 23:42:59.472 | INFO     | __main__:main:105 - Testing baseline...
2026-09-30 23:42:59.488 | INFO     | __main__:main:116 - Number of trainable parameters: 1539
2026-09-30 23:42:59.488 | INFO     | __main__:main:117 - Starting transfer learning...
2026-09-30 23:43:06.877 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.203847, Accuracy: 31.25%
2026-09-30 23:43:07.103 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.816237, Val Accuracy: 69.92%
2026-09-30 23:43:07.104 | INFO     | train:linear_probe:190 - New best validation accuracy: 69.92 at epoch 1
2026-09-30 23:43:07.167 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 0.822839, Accuracy: 71.88%
2026-09-30 23:43:07.407 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 0.568159, Val Ac

In [14]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 800 \
    --epochs 30

2026-09-30 23:52:49.629 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_beans_resnet18_seed43_epochs30.log
2026-09-30 23:52:49.633 | INFO     | __main__:main:78 - Running on cuda
2026-09-30 23:52:50.745 | INFO     | __main__:main:105 - Testing baseline...
2026-09-30 23:52:50.759 | INFO     | __main__:main:116 - Number of trainable parameters: 1539
2026-09-30 23:52:50.759 | INFO     | __main__:main:117 - Starting transfer learning...
2026-09-30 23:52:59.676 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.063459, Accuracy: 50.00%
2026-09-30 23:52:59.901 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.718151, Val Accuracy: 75.94%
2026-09-30 23:52:59.902 | INFO     | train:linear_probe:190 - New best validation accuracy: 75.94 at epoch 1
2026-09-30 23:52:59.966 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 0.651616, Accuracy: 90.62%
2026-09-30 23:53:00.210 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 0.499973, Val Ac

In [15]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 800 \
    --epochs 30

2026-10-01 00:02:46.367 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_beans_resnet18_seed44_epochs30.log
2026-10-01 00:02:46.371 | INFO     | __main__:main:78 - Running on cuda
2026-10-01 00:02:47.845 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 00:02:47.871 | INFO     | __main__:main:116 - Number of trainable parameters: 1539
2026-10-01 00:02:47.871 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 00:02:55.345 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.341334, Accuracy: 46.88%
2026-10-01 00:02:55.567 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.846501, Val Accuracy: 61.65%
2026-10-01 00:02:55.568 | INFO     | train:linear_probe:190 - New best validation accuracy: 61.65 at epoch 1
2026-10-01 00:02:55.648 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 0.886848, Accuracy: 56.25%
2026-10-01 00:02:55.875 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 0.560702, Val Ac

In [16]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 00:12:42.364 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed42_epochs30_initfixed0.78_betalr0.01.log
2026-10-01 00:12:42.435 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 00:12:43.495 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 00:12:43.495 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 00:12:43.518 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 00:12:43.519 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 00:12:43.519 | INFO     | __main__:main:439 - Total trainable parameters: 1543
2026-10-01 00:12:44.883 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.032128, Accuracy: 46.88%
2026-10-01 00:12:53.733 | INFO     | train:test_epoch:90 - Epo

In [17]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 00:17:36.114 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed43_epochs30_initfixed0.78_betalr0.01.log
2026-10-01 00:17:36.179 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 00:17:37.270 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 00:17:37.270 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 00:17:37.295 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 00:17:37.296 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 00:17:37.296 | INFO     | __main__:main:439 - Total trainable parameters: 1543
2026-10-01 00:17:38.605 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.324863, Accuracy: 28.12%
2026-10-01 00:17:46.773 | INFO     | train:test_epoch:90 - Epo

In [18]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 00:22:30.261 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed44_epochs30_initfixed0.78_betalr0.01.log
2026-10-01 00:22:30.367 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 00:22:31.554 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 00:22:31.554 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 00:22:31.579 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 00:22:31.580 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 00:22:31.580 | INFO     | __main__:main:439 - Total trainable parameters: 1543
2026-10-01 00:22:33.054 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.201310, Accuracy: 31.25%
2026-10-01 00:22:40.599 | INFO     | train:test_epoch:90 - Epo

In [19]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 00:27:23.245 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed42_epochs30_initrandom_betalr0.01.log
2026-10-01 00:27:23.348 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 00:27:24.752 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-01 00:27:24.753 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 00:27:24.793 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-10-01 00:27:24.794 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 00:27:24.794 | INFO     | __main__:main:439 - Total trainable parameters: 1543
2026-10-01 00:27:26.417 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.046550, Accuracy: 46.88%
2026-10-01 00:27:34.036 | INFO     | train:test_ep

In [20]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 00:32:16.909 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed43_epochs30_initrandom_betalr0.01.log
2026-10-01 00:32:16.980 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 00:32:18.192 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-01 00:32:18.193 | INFO     | __main__:main:394 - Requested initial stage betas: [0.889167, 0.712695, 0.705809, 0.943372]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 00:32:18.226 | INFO     | __main__:main:418 - Initial stage betas: [0.8891667127609253, 0.7126947641372681, 0.7058085799217224, 0.9433716535568237]
2026-10-01 00:32:18.227 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 00:32:18.227 | INFO     | __main__:main:439 - Total trainable parameters: 1543
2026-10-01 00:32:20.497 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.278397, Accuracy: 28.12%
2026-10-01 00:32:28.139 | INFO     | train:test_

In [21]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 00:37:08.825 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_beans_resnet18_seed44_epochs30_initrandom_betalr0.01.log
2026-10-01 00:37:08.891 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 00:37:09.924 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-01 00:37:09.924 | INFO     | __main__:main:394 - Requested initial stage betas: [0.735544, 0.774853, 0.817674, 0.981063]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 00:37:09.949 | INFO     | __main__:main:418 - Initial stage betas: [0.7355440258979797, 0.7748528122901917, 0.8176735043525696, 0.9810633659362793]
2026-10-01 00:37:09.950 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 00:37:09.950 | INFO     | __main__:main:439 - Total trainable parameters: 1543
2026-10-01 00:37:11.265 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.202614, Accuracy: 34.38%
2026-10-01 00:37:20.137 | INFO     | train:test_